# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**Christian Ayala**

Usaré 100 archivos TXT del TAR en su orden de almacenamiento, con la misma selección y tokenización de la práctica 03. La selección por orden de archivo no será una muestra aleatoria ni representativa de Gutenberg.

**Expectativa inicial:** espero que TF-IDF y BM25 compartan documentos, pero los ordenen de manera diferente. TF-IDF utilizará frecuencia cruda y normalización por la norma del vector. BM25 limitará el efecto de las repeticiones y ajustará por longitud. Compararé sus resultados sin asumir que uno será mejor para todas las consultas.


**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

**Expectativa:** espero encontrar documentos de distintas longitudes y términos con frecuencias documentales muy diferentes. Los términos raros recibirán mayor IDF. Mantendré los mismos documentos y tokens en ambos métodos para que las diferencias procedan de la puntuación.


In [1]:
import re
import math
import tarfile
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from scipy.sparse import csr_matrix

# Mostrar todas las columnas de las tablas de comparación y de aportes.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Ejecutar desde la carpeta 04 o desde la raíz de Unidad 1.
candidatos = [Path.cwd().parent / '02' / 'txt-files.tar.zip',
              Path.cwd() / '02' / 'txt-files.tar.zip']
RUTA_ZIP = next((ruta for ruta in candidatos if ruta.is_file()), None)
if RUTA_ZIP is None:
    raise FileNotFoundError('No se encontró 02/txt-files.tar.zip')

TAMANO_CORPUS = 100

# Misma regla que en las prácticas 02 y 03. Se conservan las repeticiones.
def tokenizar(texto):
    return re.findall(r'\b[a-z0-9]+\b', texto.lower())

def obtener_titulo(texto, doc_id):
    coincidencia = re.search(r'^Title:\s*(.+)$', texto, flags=re.MULTILINE)
    if coincidencia:
        return coincidencia.group(1).strip()
    coincidencia = re.search(
        r'The Project Gutenberg eBook of ([^\r\n]+)', texto, flags=re.IGNORECASE
    )
    return coincidencia.group(1).strip() if coincidencia else Path(doc_id).stem

def clave_documento(nombre):
    return re.sub(r'-\d+\.txt$', '.txt', Path(nombre).name.lower())

documentos_originales = {}
claves_vistas = set()
with zipfile.ZipFile(RUTA_ZIP) as archivo_zip:
    tar_interno = next(n for n in archivo_zip.namelist() if n.endswith('.tar'))
    # Lectura secuencial: solo se descomprimen los primeros documentos necesarios.
    with archivo_zip.open(tar_interno) as archivo:
        with tarfile.open(fileobj=archivo, mode='r|') as archivo_tar:
            for miembro in archivo_tar:
                if not miembro.isfile() or not miembro.name.lower().endswith('.txt'):
                    continue
                if clave_documento(miembro.name) in claves_vistas:
                    continue
                with archivo_tar.extractfile(miembro) as texto_archivo:
                    documentos_originales[miembro.name] = texto_archivo.read().decode(
                        'utf-8', errors='ignore'
                    )
                claves_vistas.add(clave_documento(miembro.name))
                if len(documentos_originales) == TAMANO_CORPUS:
                    break

assert len(documentos_originales) == TAMANO_CORPUS

tabla_documentos = pd.DataFrame([
    {'doc_id': str(nombre), 'Título': obtener_titulo(texto, nombre),
     'Texto': texto, 'tokens': tokenizar(texto)}
    for nombre, texto in documentos_originales.items()
]).sort_values('doc_id').reset_index(drop=True)

nombres_documentos = tabla_documentos['doc_id'].tolist()
titulos = dict(zip(nombres_documentos, tabla_documentos['Título']))
N = len(tabla_documentos)
tf_documentos = [Counter(tokens) for tokens in tabla_documentos['tokens']]
longitudes = np.array([len(tokens) for tokens in tabla_documentos['tokens']])
avgdl = float(longitudes.mean())

df_terminos = Counter()
for frecuencias in tf_documentos:
    df_terminos.update(frecuencias.keys())

# Las dos IDF se calculan una sola vez, exclusivamente con el corpus.
idf_tfidf = {t: math.log(N / df) + 1 for t, df in df_terminos.items()}
idf_bm25 = {
    t: math.log(1 + (N - df + 0.5) / (df + 0.5))
    for t, df in df_terminos.items()
}

display(tabla_documentos)
display(pd.DataFrame([{
    'Documentos': N, 'Vocabulario': len(df_terminos),
    'Longitud mínima': int(longitudes.min()), 'Longitud media': avgdl,
    'Longitud máxima': int(longitudes.max())
}]))


,doc_id,Título,Texto,tokens
0,cache/epub/1000/pg1000.txt,La Divina Commedia di Dante: Complete,The Project Gutenberg eBook of La Divina Commedia di Dante: Complete\r\n \r\nThis eBook is fo...,"[the, project, gutenberg, ebook, of, la, divina, commedia, di, dante, complete, this, ebook, is,..."
1,cache/epub/10000/pg10000.txt,The Magna Carta,The Project Gutenberg eBook of The Magna Carta\r\n \r\nThis eBook is for the use of anyone an...,"[the, project, gutenberg, ebook, of, the, magna, carta, this, ebook, is, for, the, use, of, anyo..."
2,cache/epub/10001/pg10001.txt,Apocolocyntosis,The Project Gutenberg eBook of Apocolocyntosis\r\n \r\nThis eBook is for the use of anyone an...,"[the, project, gutenberg, ebook, of, apocolocyntosis, this, ebook, is, for, the, use, of, anyone..."
3,cache/epub/10002/pg10002.txt,The House on the Borderland,The Project Gutenberg eBook of The House on the Borderland\r\n \r\nThis eBook is for the use ...,"[the, project, gutenberg, ebook, of, the, house, on, the, borderland, this, ebook, is, for, the,..."
4,cache/epub/10003/pg10003.txt,"My First Years as a Frenchwoman, 1876-1879","The Project Gutenberg eBook of My First Years as a Frenchwoman, 1876-1879\r\n \r\nThis eBook ...","[the, project, gutenberg, ebook, of, my, first, years, as, a, frenchwoman, 1876, 1879, this, ebo..."
...,...,...,...,...
95,cache/epub/10086/pg10086.txt,The Minute Boys of the Mohawk Valley,The Project Gutenberg eBook of The Minute Boys of the Mohawk Valley\r\n \r\nThis eBook is for...,"[the, project, gutenberg, ebook, of, the, minute, boys, of, the, mohawk, valley, this, ebook, is..."
96,cache/epub/10087/pg10087.txt,Old Lady Number 31,The Project Gutenberg eBook of Old Lady Number 31\r\n \r\nThis eBook is for the use of anyone...,"[the, project, gutenberg, ebook, of, old, lady, number, 31, this, ebook, is, for, the, use, of, ..."
97,cache/epub/10088/pg10088.txt,"Thaumaturgia; Or, Elucidations of the Marvellous","The Project Gutenberg eBook of Thaumaturgia; Or, Elucidations of the Marvellous\r\n \r\nThis ...","[the, project, gutenberg, ebook, of, thaumaturgia, or, elucidations, of, the, marvellous, this, ..."
98,cache/epub/10089/pg10089.txt,Elves and Heroes,The Project Gutenberg eBook of Elves and Heroes\r\n \r\nThis eBook is for the use of anyone a...,"[the, project, gutenberg, ebook, of, elves, and, heroes, this, ebook, is, for, the, use, of, any..."


,Documentos,Vocabulario,Longitud mínima,Longitud media,Longitud máxima
0,100,98167,5400,57156.29,182221


**Preprocesamiento:** usaré minúsculas y secuencias ASCII de letras y números, sin eliminar stopwords ni aplicar stemming o lematización. Los encabezados y licencias se conservan y pueden introducir coincidencias ajenas al contenido. La deduplicación por nombre evita algunas versiones de un mismo identificador, pero no detecta obras equivalentes con IDs distintos.

**Pesos utilizados:**

- TF-IDF: $w_{t,d}=f(t,d)[\ln(N/df(t))+1]$, igual que en 03.
- BM25: $IDF(t)=\ln[1+(N-df(t)+0.5)/(df(t)+0.5)]$.

Las estadísticas se calculan solo con la colección. Las consultas no se añaden como documentos.


In [2]:
tabla_estadisticas = pd.DataFrame([
    {'Término': t, 'DF': df_terminos[t],
     'IDF TF-IDF': idf_tfidf[t], 'IDF BM25': idf_bm25[t]}
    for t in sorted(df_terminos)
])
print('Términos con mayor frecuencia documental:')
display(tabla_estadisticas.sort_values(['DF', 'Término'], ascending=[False, True]).head(10))
print('Ejemplos de términos poco frecuentes:')
display(tabla_estadisticas.sort_values(['DF', 'Término']).head(10))


Términos con mayor frecuencia documental:


,Término,DF,IDF TF-IDF,IDF BM25
2,000,100,1.0,0.004963
59,07042,100,1.0,0.004963
80,1,100,1.0,0.004963
839,2,100,1.0,0.004963
840,20,100,1.0,0.004963
845,2001,100,1.0,0.004963
1001,3,100,1.0,0.004963
1002,30,100,1.0,0.004963
1133,4,100,1.0,0.004963
1149,41,100,1.0,0.004963


Ejemplos de términos poco frecuentes:


,Término,DF,IDF TF-IDF,IDF BM25
4,0014,1,5.60517,4.209655
5,0023,1,5.60517,4.209655
7,0035,1,5.60517,4.209655
8,004,1,5.60517,4.209655
9,0042,1,5.60517,4.209655
10,005,1,5.60517,4.209655
11,0055,1,5.60517,4.209655
12,006,1,5.60517,4.209655
13,007,1,5.60517,4.209655
14,008,1,5.60517,4.209655


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

**Expectativa:** espero que una consulta frecuente recupere muchos documentos y una rara produzca listas más cortas. En BM25, repetir mucho un término no aumentará la puntuación de forma proporcional. Usaré ocho consultas de una a cuatro palabras y justificaré su frecuencia mediante DF.


**TF-IDF + coseno:** construimos una matriz CSR con los pesos y normalizamos sus filas
usando la norma L2. La consulta usa los mismos ejes e IDF. El producto de los vectores
normalizados es el coseno. La matriz documento-término permanece dispersa.

**BM25:** para los términos distintos de la consulta aplicamos

$$s(q,d)=\sum_{t\in\operatorname{set}(q)} IDF(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/avgdl)}.$$

Un término ausente aporta cero. No añadimos la consulta a la colección y no contamos
varias veces un término repetido de la consulta en BM25, siguiendo los slides.


In [3]:
vocabulario = sorted(df_terminos)
indice_terminos = {termino: j for j, termino in enumerate(vocabulario)}

filas, columnas, valores = [], [], []
for i, frecuencias in enumerate(tf_documentos):
    for termino, frecuencia in frecuencias.items():
        filas.append(i)
        columnas.append(indice_terminos[termino])
        valores.append(frecuencia * idf_tfidf[termino])

matriz_tfidf = csr_matrix(
    (valores, (filas, columnas)), shape=(N, len(vocabulario)), dtype=float
)
normas_documentos = np.sqrt(matriz_tfidf.multiply(matriz_tfidf).sum(axis=1)).A1
inversas = np.divide(1.0, normas_documentos, out=np.zeros(N), where=normas_documentos > 0)
matriz_normalizada = matriz_tfidf.multiply(inversas[:, None]).tocsr()

def puntuar_tfidf(consulta):
    frecuencias = Counter(tokenizar(consulta))
    terminos = [t for t in frecuencias if t in indice_terminos]
    vector_q = csr_matrix((
        [frecuencias[t] * idf_tfidf[t] for t in terminos],
        ([0] * len(terminos), [indice_terminos[t] for t in terminos])
    ), shape=(1, len(vocabulario)))
    norma_q = math.sqrt(float(vector_q.multiply(vector_q).sum()))
    if norma_q == 0:
        return np.zeros(N)
    # Solo convertimos a denso el vector de N puntuaciones, no la matriz del corpus.
    return (matriz_normalizada @ (vector_q / norma_q).T).toarray().ravel()

def aporte_bm25(termino, indice_documento, k1=1.2, b=0.75):
    frecuencia = tf_documentos[indice_documento].get(termino, 0)
    if frecuencia == 0:
        return 0.0
    ajuste = k1 * (1 - b + b * longitudes[indice_documento] / avgdl)
    return idf_bm25[termino] * frecuencia * (k1 + 1) / (frecuencia + ajuste)

def puntuar_bm25(consulta, k1=1.2, b=0.75):
    terminos = sorted(set(tokenizar(consulta)) & df_terminos.keys())
    return np.array([
        sum(aporte_bm25(t, i, k1, b) for t in terminos)
        for i in range(N)
    ])

def obtener_ranking(puntuaciones, top_k=10):
    # El ranking completo también permite analizar posiciones fuera del top 10.
    indices = sorted(
        (i for i, score in enumerate(puntuaciones) if score > 0),
        key=lambda i: (-float(puntuaciones[i]), str(nombres_documentos[i]))
    )
    if top_k is not None:
        indices = indices[:top_k]
    return pd.DataFrame([
        {'Posición': posicion, 'doc_id': nombres_documentos[i],
         'Título': titulos[nombres_documentos[i]], 'Score': float(puntuaciones[i])}
        for posicion, i in enumerate(indices, start=1)
    ], columns=['Posición', 'doc_id', 'Título', 'Score'])

print('Matriz TF-IDF:', matriz_tfidf.shape, '| valores no nulos:', matriz_tfidf.nnz)
print('Formato:', matriz_tfidf.format)


Matriz TF-IDF: (100, 98167) | valores no nulos: 603074
Formato: csr


In [4]:
# Consultas en inglés. incluyen distintas longitudes y frecuencias documentales.
consultas = {
    'q1': ('the', 'Examinar una palabra funcional muy frecuente como control léxico. no expresa una necesidad temática precisa.'),
    'q2': ('love', 'Encontrar textos que traten el amor.'),
    'q3': ('history england', 'Encontrar textos sobre historia de Inglaterra.'),
    'q4': ('science mathematics', 'Encontrar textos sobre ciencia y matemáticas.'),
    'q5': ('sea voyage ship', 'Encontrar relatos de navegación y viajes marítimos.'),
    'q6': ('magna carta', 'Encontrar textos que mencionen la Magna Carta.'),
    'q7': ('apocolocyntosis', 'Localizar menciones de la obra Apocolocyntosis.'),
    'q8': ('war military king battle', 'Encontrar textos sobre guerras, reyes y actividad militar.')
}

filas_consultas = []
for qid, (texto, necesidad) in consultas.items():
    terminos = sorted(set(tokenizar(texto)))
    filas_consultas.append({
        'Consulta': qid, 'Texto': texto, 'Qué busca el usuario': necesidad,
        'Cantidad de tokens': len(tokenizar(texto)),
        'DF por término': '. '.join(f'{t}: {df_terminos.get(t, 0)}/{N}' for t in terminos)
    })
display(pd.DataFrame(filas_consultas))

rankings_tfidf, rankings_bm25 = {}, {}
scores_tfidf, scores_bm25 = {}, {}
for qid, (texto, _) in consultas.items():
    scores_tfidf[qid] = puntuar_tfidf(texto)
    scores_bm25[qid] = puntuar_bm25(texto)
    rankings_tfidf[qid] = obtener_ranking(scores_tfidf[qid])
    rankings_bm25[qid] = obtener_ranking(scores_bm25[qid])

print('Justificación de frecuencias:')
for termino in ['the', 'love', 'magna', 'carta', 'apocolocyntosis']:
    print(f'{termino}: DF={df_terminos.get(termino, 0)}/{N}')
print('Un DF mayor indica presencia en más libros, no más apariciones dentro de un libro.')
print('Las consultas tienen entre 1 y 4 tokens. q1 es un control de alta frecuencia.')


,Consulta,Texto,Qué busca el usuario,Cantidad de tokens,DF por término
0,q1,the,Examinar una palabra funcional muy frecuente como control léxico. no expresa una necesidad temát...,1,the: 100/100
1,q2,love,Encontrar textos que traten el amor.,1,love: 85/100
2,q3,history england,Encontrar textos sobre historia de Inglaterra.,2,england: 67/100. history: 72/100
3,q4,science mathematics,Encontrar textos sobre ciencia y matemáticas.,2,mathematics: 10/100. science: 56/100
4,q5,sea voyage ship,Encontrar relatos de navegación y viajes marítimos.,3,sea: 86/100. ship: 63/100. voyage: 37/100
5,q6,magna carta,Encontrar textos que mencionen la Magna Carta.,2,carta: 3/100. magna: 9/100
6,q7,apocolocyntosis,Localizar menciones de la obra Apocolocyntosis.,1,apocolocyntosis: 1/100
7,q8,war military king battle,"Encontrar textos sobre guerras, reyes y actividad militar.",4,battle: 63/100. king: 75/100. military: 49/100. war: 76/100


Justificación de frecuencias:
the: DF=100/100
love: DF=85/100
magna: DF=9/100
carta: DF=3/100
apocolocyntosis: DF=1/100
Un DF mayor indica presencia en más libros, no más apariciones dentro de un libro.
Las consultas tienen entre 1 y 4 tokens. q1 es un control de alta frecuencia.


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

**Expectativa:** espero diferencias de posición aunque los top 10 compartan varios documentos. Analizaré `sea voyage ship` para relacionar esas diferencias con la frecuencia, la longitud y la IDF. Compararé los scores dentro de cada método, porque el coseno y BM25 tienen escalas distintas.


In [5]:
resumen = []
for qid, (texto, _) in consultas.items():
    tf = rankings_tfidf[qid]
    bm = rankings_bm25[qid]
    # Unión de posiciones para no perder filas si las listas tienen tamaños diferentes.
    izquierda = tf.assign(
        **{'ID y título TF-IDF': tf['doc_id'] + ' | ' + tf['Título']}
    )[['Posición', 'ID y título TF-IDF', 'Score']].rename(columns={'Score': 'Coseno'})
    derecha = bm.assign(
        **{'ID y título BM25': bm['doc_id'] + ' | ' + bm['Título']}
    )[['Posición', 'ID y título BM25', 'Score']].rename(columns={'Score': 'Score BM25'})
    comparacion = izquierda.merge(derecha, on='Posición', how='outer').sort_values('Posición')
    print(f'\n{qid}: {texto}')
    display(comparacion)
    resumen.append({
        'Consulta': qid, 'Tamaño TF-IDF': len(tf), 'Tamaño BM25': len(bm),
        'IDs compartidos': len(set(tf['doc_id']) & set(bm['doc_id'])),
        'Primero TF-IDF': tf.iloc[0]['doc_id'] if len(tf) else None,
        'Primero BM25': bm.iloc[0]['doc_id'] if len(bm) else None
    })
tabla_resumen = pd.DataFrame(resumen)
print('Resumen de los top 10 (o menos, si no hay diez scores positivos):')
display(tabla_resumen)



q1: the


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,cache/epub/10071/pg10071.txt | By-Ways of Bombay,0.744277,cache/epub/10058/pg10058.txt | The Divine Office: A Study of the Roman Breviary,0.010916
1,2,cache/epub/10058/pg10058.txt | The Divine Office: A Study of the Roman Breviary,0.736734,cache/epub/10060/pg10060.txt | Discourses: Biological & Geological,0.010916
2,3,cache/epub/10012/pg10012.txt | The Mountains of California,0.719695,cache/epub/10012/pg10012.txt | The Mountains of California,0.010916
3,4,cache/epub/10065/pg10065.txt | The Constitution of the United States,0.716166,cache/epub/10090/pg10090.txt | Peaceless Europe,0.010916
4,5,cache/epub/10038/pg10038.txt | The Magnetic North,0.704036,"cache/epub/10088/pg10088.txt | Thaumaturgia; Or, Elucidations of the Marvellous",0.010916
5,6,cache/epub/10060/pg10060.txt | Discourses: Biological & Geological,0.703879,cache/epub/10038/pg10038.txt | The Magnetic North,0.010916
6,7,cache/epub/10043/pg10043.txt | Klondyke Nuggets: A Brief Description of the Great Gold Regions i...,0.702454,cache/epub/10062/pg10062.txt | The Iron Game: A Tale of the War,0.010915
7,8,cache/epub/10080/pg10080.txt | Mobilizing Woman-Power,0.700451,"cache/epub/10009/pg10009.txt | Wild Northern Scenes; Or, Sporting Adventures with the Rifle and ...",0.010915
8,9,cache/epub/10062/pg10062.txt | The Iron Game: A Tale of the War,0.699689,cache/epub/10085/pg10085.txt | Moorish Literature,0.010915
9,10,cache/epub/10089/pg10089.txt | Elves and Heroes,0.696441,cache/epub/10084/pg10084.txt | Kazan,0.010915



q2: love


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,cache/epub/10063/pg10063.txt | Happiness and Marriage,0.047649,"cache/epub/10039/pg10039.txt | The Works of Aphra Behn, Volume III",0.364578
1,2,"cache/epub/10039/pg10039.txt | The Works of Aphra Behn, Volume III",0.043636,cache/epub/10063/pg10063.txt | Happiness and Marriage,0.364006
2,3,cache/epub/10006/pg10006.txt | La Fiammetta,0.041653,cache/epub/10085/pg10085.txt | Moorish Literature,0.363946
3,4,cache/epub/10085/pg10085.txt | Moorish Literature,0.029245,cache/epub/10064/pg10064.txt | Beltane the Smith,0.363457
4,5,"cache/epub/1007/pg1007.txt | Divine Comedy, Cary's Translation, Paradise",0.028484,cache/epub/10006/pg10006.txt | La Fiammetta,0.362969
5,6,"cache/epub/1003/pg1003.txt | Divine Comedy, Longfellow's Translation, Paradise",0.026647,"cache/epub/1007/pg1007.txt | Divine Comedy, Cary's Translation, Paradise",0.362810
6,7,cache/epub/10064/pg10064.txt | Beltane the Smith,0.021370,"cache/epub/1003/pg1003.txt | Divine Comedy, Longfellow's Translation, Paradise",0.362798
7,8,cache/epub/10031/pg10031.txt | The Complete Poetical Works of Edgar Allan Poe,0.020673,cache/epub/10031/pg10031.txt | The Complete Poetical Works of Edgar Allan Poe,0.362582
8,9,cache/epub/10037/pg10037.txt | A Beautiful Possibility,0.019586,cache/epub/10037/pg10037.txt | A Beautiful Possibility,0.362026
9,10,"cache/epub/1006/pg1006.txt | Divine Comedy, Cary's Translation, Purgatory",0.017996,"cache/epub/1004/pg1004.txt | Divine Comedy, Longfellow's Translation, Complete",0.361899



q3: history england


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,cache/epub/10000/pg10000.txt | The Magna Carta,0.017743,cache/epub/10065/pg10065.txt | The Constitution of the United States,1.570781
1,2,cache/epub/10065/pg10065.txt | The Constitution of the United States,0.012997,cache/epub/10044/pg10044.txt | The Father of British Canada: a Chronicle of Carleton,1.529472
2,3,cache/epub/10044/pg10044.txt | The Father of British Canada: a Chronicle of Carleton,0.009255,"cache/epub/10077/pg10077.txt | The Atlantic Monthly, Volume 10, No. 60, October, 1862",1.526995
3,4,cache/epub/10069/pg10069.txt | Account of the Romansh Language,0.007366,cache/epub/10090/pg10090.txt | Peaceless Europe,1.519842
4,5,cache/epub/10042/pg10042.txt | The child under eight,0.006768,cache/epub/10042/pg10042.txt | The child under eight,1.515055
5,6,"cache/epub/10077/pg10077.txt | The Atlantic Monthly, Volume 10, No. 60, October, 1862",0.006651,"cache/epub/10088/pg10088.txt | Thaumaturgia; Or, Elucidations of the Marvellous",1.475774
6,7,cache/epub/10024/pg10024.txt | Beneath the Banner: Being Narratives of Noble Lives and Brave Deeds,0.006496,"cache/epub/10079/pg10079.txt | The Atlantic Monthly, Volume 02, No. 09, July, 1858",1.466851
7,8,cache/epub/10060/pg10060.txt | Discourses: Biological & Geological,0.006099,cache/epub/10060/pg10060.txt | Discourses: Biological & Geological,1.458658
8,9,cache/epub/10004/pg10004.txt | The Warriors,0.005886,cache/epub/10070/pg10070.txt | Stories of Great Americans for Little Americans,1.451853
9,10,cache/epub/10078/pg10078.txt | Plain Words from America: A Letter to a German Professor (1917),0.005759,cache/epub/10078/pg10078.txt | Plain Words from America: A Letter to a German Professor (1917),1.446684


... (8 outputs left out)


q8: war military king battle


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,cache/epub/10078/pg10078.txt | Plain Words from America: A Letter to a German Professor (1917),0.051825,cache/epub/10044/pg10044.txt | The Father of British Canada: a Chronicle of Carleton,3.681228
1,2,cache/epub/10090/pg10090.txt | Peaceless Europe,0.045158,cache/epub/10086/pg10086.txt | The Minute Boys of the Mohawk Valley,3.567906
2,3,cache/epub/10080/pg10080.txt | Mobilizing Woman-Power,0.024797,cache/epub/10024/pg10024.txt | Beneath the Banner: Being Narratives of Noble Lives and Brave Deeds,3.455551
3,4,cache/epub/10044/pg10044.txt | The Father of British Canada: a Chronicle of Carleton,0.019860,cache/epub/10062/pg10062.txt | The Iron Game: A Tale of the War,3.437723
4,5,cache/epub/10085/pg10085.txt | Moorish Literature,0.017164,cache/epub/10090/pg10090.txt | Peaceless Europe,3.252236
5,6,cache/epub/10056/pg10056.txt | Chinese literature,0.016357,"cache/epub/10079/pg10079.txt | The Atlantic Monthly, Volume 02, No. 09, July, 1858",3.208210
6,7,"cache/epub/10077/pg10077.txt | The Atlantic Monthly, Volume 10, No. 60, October, 1862",0.014889,"cache/epub/10030/pg10030.txt | The Life of Hon. William F. Cody, Known as Buffalo Bill, the Famo...",3.175077
7,8,cache/epub/10000/pg10000.txt | The Magna Carta,0.013705,cache/epub/10073/pg10073.txt | The Century Vocabulary Builder,3.150011
8,9,cache/epub/10062/pg10062.txt | The Iron Game: A Tale of the War,0.013385,"cache/epub/10035/pg10035.txt | Punchinello, Volume 2, No. 27, October 1, 1870",3.126598
9,10,"cache/epub/10081/pg10081.txt | The Boy Allies at Jutland; Or, The Greatest Naval Battle of History",0.012542,cache/epub/10004/pg10004.txt | The Warriors,3.119836


Resumen de los top 10 (o menos, si no hay diez scores positivos):


,Consulta,Tamaño TF-IDF,Tamaño BM25,IDs compartidos,Primero TF-IDF,Primero BM25
0,q1,10,10,5,cache/epub/10071/pg10071.txt,cache/epub/10058/pg10058.txt
1,q2,10,10,9,cache/epub/10063/pg10063.txt,cache/epub/10039/pg10039.txt
2,q3,10,10,6,cache/epub/10000/pg10000.txt,cache/epub/10065/pg10065.txt
3,q4,10,10,6,cache/epub/10060/pg10060.txt,cache/epub/10042/pg10042.txt
4,q5,10,10,8,cache/epub/10008/pg10008.txt,cache/epub/10005/pg10005.txt
5,q6,10,10,10,cache/epub/10000/pg10000.txt,cache/epub/10000/pg10000.txt
6,q7,1,1,1,cache/epub/10001/pg10001.txt,cache/epub/10001/pg10001.txt
7,q8,10,10,3,cache/epub/10078/pg10078.txt,cache/epub/10044/pg10044.txt


In [6]:
# Comparar dos libros de navegación para la misma consulta.
consulta_elegida = 'q5'
doc_a = next(doc_id for doc_id in nombres_documentos if titulos[doc_id] == 'The Mystery')
doc_b = next(doc_id for doc_id in nombres_documentos if titulos[doc_id] == 'A Voyage to the Moon')
texto_elegido = consultas[consulta_elegida][0]
terminos_elegidos = sorted(set(tokenizar(texto_elegido)))
indices_elegidos = [nombres_documentos.index(doc_a), nombres_documentos.index(doc_b)]

ranking_tf_completo = obtener_ranking(scores_tfidf[consulta_elegida], None)
ranking_bm_completo = obtener_ranking(scores_bm25[consulta_elegida], None)
pos_tf = dict(zip(ranking_tf_completo['doc_id'], ranking_tf_completo['Posición']))
pos_bm = dict(zip(ranking_bm_completo['doc_id'], ranking_bm_completo['Posición']))

detalle = []
for i in indices_elegidos:
    for t in terminos_elegidos:
        f = tf_documentos[i].get(t, 0)
        ajuste = 1.2 * (1 - 0.75 + 0.75 * longitudes[i] / avgdl)
        detalle.append({
            'doc_id': nombres_documentos[i], 'Término': t, '|d|': int(longitudes[i]),
            '|d| / avgdl': longitudes[i] / avgdl, 'f(t,d)': f,
            'DF': df_terminos.get(t, 0), 'IDF BM25': idf_bm25.get(t, 0),
            'Factor TF saturado': f * 2.2 / (f + ajuste) if f else 0,
            'Aporte BM25': aporte_bm25(t, i)
        })
print('Consulta seleccionada:', consulta_elegida, texto_elegido)
print('Longitud media:', avgdl)
display(pd.DataFrame(detalle))

# Descomponer también el coseno para explicar su propio orden.
conteo_q = Counter(tokenizar(texto_elegido))
pesos_q = {t: conteo_q[t] * idf_tfidf[t] for t in terminos_elegidos if t in idf_tfidf}
norma_q = math.sqrt(sum(peso ** 2 for peso in pesos_q.values()))
comparacion_documentos = []
for i in indices_elegidos:
    doc_id = nombres_documentos[i]
    punto = sum(peso * tf_documentos[i].get(t, 0) * idf_tfidf[t] for t, peso in pesos_q.items())
    comparacion_documentos.append({
        'doc_id': doc_id, 'Título': titulos[doc_id], '|d|': int(longitudes[i]),
        'Posición TF-IDF': pos_tf[doc_id], 'Coseno': scores_tfidf[consulta_elegida][i],
        'Producto punto': punto, 'Norma consulta': norma_q,
        'Norma documento': normas_documentos[i],
        'Posición BM25': pos_bm[doc_id], 'Score BM25': scores_bm25[consulta_elegida][i]
    })
display(pd.DataFrame(comparacion_documentos))
print(f'TF-IDF ordena {doc_a} antes que {doc_b}. BM25 invierte ese orden.')
for fila in comparacion_documentos:
    print(f"{fila['doc_id']}: coseno = {fila['Producto punto']:.6f} / "
          f"({norma_q:.6f} × {fila['Norma documento']:.6f}) = {fila['Coseno']:.6f}.")
print('BM25 ordena por la suma de los aportes mostrados. sus valores no se comparan con la escala del coseno.')


Consulta seleccionada: q5 sea voyage ship
Longitud media: 57156.29


,doc_id,Término,|d|,|d| / avgdl,"f(t,d)",DF,IDF BM25,Factor TF saturado,Aporte BM25
0,cache/epub/10008/pg10008.txt,sea,70986,1.241963,79,86,0.154976,2.161214,0.334937
1,cache/epub/10008/pg10008.txt,ship,70986,1.241963,118,63,0.464081,2.173881,1.008856
2,cache/epub/10008/pg10008.txt,voyage,70986,1.241963,11,37,0.990780,1.948821,1.930852
3,cache/epub/10005/pg10005.txt,sea,74098,1.296410,13,86,0.154976,1.976945,0.306379
4,cache/epub/10005/pg10005.txt,ship,74098,1.296410,13,63,0.464081,1.976945,0.917462
5,cache/epub/10005/pg10005.txt,voyage,74098,1.296410,60,37,0.990780,2.147502,2.127701


,doc_id,Título,|d|,Posición TF-IDF,Coseno,Producto punto,Norma consulta,Norma documento,Posición BM25,Score BM25
0,cache/epub/10008/pg10008.txt,The Mystery,70986,1,0.020212,400.605163,2.72745,7267.071043,2,3.274644
1,cache/epub/10005/pg10005.txt,A Voyage to the Moon,74098,3,0.013062,283.627761,2.72745,7961.475686,1,3.351542


TF-IDF ordena cache/epub/10008/pg10008.txt antes que cache/epub/10005/pg10005.txt. BM25 invierte ese orden.
cache/epub/10008/pg10008.txt: coseno = 400.605163 / (2.727450 × 7267.071043) = 0.020212.
cache/epub/10005/pg10005.txt: coseno = 283.627761 / (2.727450 × 7961.475686) = 0.013062.
BM25 ordena por la suma de los aportes mostrados. sus valores no se comparan con la escala del coseno.


**Interpretación:** para `sea voyage ship`, *The Mystery* pasa de la posición 1 en TF-IDF a la 2 en BM25. *A Voyage to the Moon* pasa de la 3 a la 1.

*The Mystery* contiene más apariciones de `sea` y `ship`, pero la saturación limita su ventaja conjunta en BM25 a unos 0,1200 puntos. *A Voyage to the Moon* contiene `voyage` 60 veces frente a 11. Como este término tiene menor DF, su ventaja de unos 0,1968 puntos supera la anterior. Así alcanza BM25 de 3,351542 frente a 3,274644, aunque es algo más largo.

TF-IDF favorece a *The Mystery*: tiene mayor producto punto y menor norma documental, con la misma norma de consulta. Su coseno es 0,020212 frente a 0,013062. Las tablas permiten comprobar estos componentes por separado.

Los top 10 de esta consulta comparten ocho IDs. La intersección mide coincidencia de documentos, no acuerdo de posiciones ni calidad. Sin juicios de relevancia no se puede concluir que BM25 recupere mejor.


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

**Expectativa:** al poner $b=0$, espero que los documentos largos dejen de recibir penalización por longitud y que los cortos pierdan esa ventaja. Al aumentar $k_1$ a 2, espero que las repeticiones tengan más efecto antes de saturarse. Los scores podrán cambiar sin que necesariamente cambien las posiciones.


In [7]:
# Reutilizamos la consulta analizada y buscamos dos documentos que coincidan
# pero tengan distintas longitudes y frecuencias para al menos un término.
candidatos = [i for i in range(N) if scores_bm25[consulta_elegida][i] > 0]
pares = [
    (i, j) for i in candidatos for j in candidatos if i < j
    and longitudes[i] != longitudes[j]
    and any(tf_documentos[i].get(t, 0) > 0 and tf_documentos[j].get(t, 0) > 0
            and tf_documentos[i][t] != tf_documentos[j][t] for t in terminos_elegidos)
]
# La mayor diferencia de longitud facilita observar el ajuste respecto a avgdl.
par_parametros = max(pares, key=lambda par: abs(int(longitudes[par[0]]) - int(longitudes[par[1]])))
configuraciones = [('Base', 1.2, 0.75), ('Sin ajuste por longitud', 1.2, 0.0),
                   ('Mayor efecto de repeticiones', 2.0, 0.75)]
filas_parametros, aportes_parametros = [], []
for nombre, k1, b in configuraciones:
    scores = puntuar_bm25(texto_elegido, k1, b)
    ranking = obtener_ranking(scores, top_k=None)
    posiciones = dict(zip(ranking['doc_id'], ranking['Posición']))
    for i in par_parametros:
        doc_id = nombres_documentos[i]
        filas_parametros.append({
            'Configuración': nombre, 'k1': k1, 'b': b, 'doc_id': doc_id,
            'Título': titulos[doc_id], '|d|': int(longitudes[i]),
            '|d| / avgdl': longitudes[i] / avgdl,
            'Posición': posiciones[doc_id], 'Score BM25': scores[i]
        })
        for t in terminos_elegidos:
            aportes_parametros.append({
                'Configuración': nombre, 'doc_id': doc_id, 'Término': t,
                'f(t,d)': tf_documentos[i].get(t, 0), 'DF': df_terminos.get(t, 0),
                'Aporte BM25': aporte_bm25(t, i, k1, b)
            })
tabla_parametros = pd.DataFrame(filas_parametros)
tabla_aportes = pd.DataFrame(aportes_parametros)
print('Consulta fija:', texto_elegido, '| avgdl fija:', avgdl)
display(tabla_parametros)
display(tabla_aportes.pivot(index=['doc_id', 'Término', 'f(t,d)', 'DF'],
                          columns='Configuración', values='Aporte BM25').reset_index())

for i in par_parametros:
    doc_id = nombres_documentos[i]
    datos = tabla_parametros[tabla_parametros['doc_id'] == doc_id].set_index('Configuración')
    base = datos.loc['Base']
    sin_longitud = datos.loc['Sin ajuste por longitud']
    mayor_k1 = datos.loc['Mayor efecto de repeticiones']
    relacion = 'más corto' if longitudes[i] < avgdl else 'más largo'
    print(f'\n{doc_id} es {relacion} que la media: {longitudes[i]} frente a {avgdl:.2f} tokens.')
    print(f"Al quitar el ajuste: score {base['Score BM25']:.6f} → "
          f"{sin_longitud['Score BM25']:.6f}. posición {int(base['Posición'])} → "
          f"{int(sin_longitud['Posición'])}.")
    print(f"Al aumentar k1: score {base['Score BM25']:.6f} → {mayor_k1['Score BM25']:.6f}. "
          f"posición {int(base['Posición'])} → {int(mayor_k1['Posición'])}.")
    if datos['Posición'].nunique() == 1:
        print('La posición se mantiene: los cambios de score no bastaron para cruzar a otros documentos.')


Consulta fija: sea voyage ship | avgdl fija: 57156.29


,Configuración,k1,b,doc_id,Título,|d|,|d| / avgdl,Posición,Score BM25
0,Base,1.2,0.75,cache/epub/10040/pg10040.txt,Aleutian Indian and English Dictionary,5400,0.094478,71,0.316570
1,Base,1.2,0.75,cache/epub/10064/pg10064.txt,Beltane the Smith,182221,3.188118,89,0.165795
2,Sin ajuste por longitud,1.2,0.00,cache/epub/10040/pg10040.txt,Aleutian Indian and English Dictionary,5400,0.094478,72,0.274958
3,Sin ajuste por longitud,1.2,0.00,cache/epub/10064/pg10064.txt,Beltane the Smith,182221,3.188118,78,0.243534
4,Mayor efecto de repeticiones,2.0,0.75,cache/epub/10040/pg10040.txt,Aleutian Indian and English Dictionary,5400,0.094478,71,0.412045
5,Mayor efecto de repeticiones,2.0,0.75,cache/epub/10064/pg10064.txt,Beltane the Smith,182221,3.188118,89,0.168408


Configuración,doc_id,Término,"f(t,d)",DF,Base,Mayor efecto de repeticiones,Sin ajuste por longitud
0,cache/epub/10040/pg10040.txt,sea,5,86,0.316570,0.412045,0.274958
1,cache/epub/10040/pg10040.txt,ship,0,63,0.000000,0.000000,0.000000
2,cache/epub/10040/pg10040.txt,voyage,0,37,0.000000,0.000000,0.000000
3,cache/epub/10064/pg10064.txt,sea,3,86,0.165795,0.168408,0.243534
4,cache/epub/10064/pg10064.txt,ship,0,63,0.000000,0.000000,0.000000
5,cache/epub/10064/pg10064.txt,voyage,0,37,0.000000,0.000000,0.000000



cache/epub/10040/pg10040.txt es más corto que la media: 5400 frente a 57156.29 tokens.
Al quitar el ajuste: score 0.316570 → 0.274958. posición 71 → 72.
Al aumentar k1: score 0.316570 → 0.412045. posición 71 → 71.



cache/epub/10064/pg10064.txt es más largo que la media: 182221 frente a 57156.29 tokens.
Al quitar el ajuste: score 0.165795 → 0.243534. posición 89 → 78.
Al aumentar k1: score 0.165795 → 0.168408. posición 89 → 89.


**Interpretación de los parámetros:** al poner $b=0$, el documento de 5.400 tokens pasa del puesto 71 al 72 y pierde la ventaja de ser más corto que la media de 57.156,29 tokens. *Beltane the Smith*, de 182.221 tokens, pasa del 89 al 78 al desaparecer su penalización por longitud.

Con $k_1=2$, sus scores cambian de 0,316570 a 0,412045 y de 0,165795 a 0,168408, respectivamente. Mantienen los puestos 71 y 89 porque los nuevos scores no cruzan los de otros documentos. Las posiciones corresponden al ranking completo de documentos con score positivo.

Un $k_1$ mayor retrasa la saturación, pero no garantiza que todos los aportes aumenten. El efecto depende también de la frecuencia y de la longitud del documento. La tabla de aportes permite comprobarlo manteniendo fijos la colección y el preprocesamiento.


**Comprobación final:** se verifican el cálculo del coseno, la suma de aportes BM25, los desempates y las consultas sin términos en el vocabulario.

In [8]:
for fila in comparacion_documentos:
    coseno_explicito = fila['Producto punto'] / (fila['Norma consulta'] * fila['Norma documento'])
    assert np.isclose(coseno_explicito, fila['Coseno'], atol=1e-12)
assert matriz_tfidf.format == 'csr' and matriz_normalizada.format == 'csr'
assert obtener_ranking(np.ones(N))['doc_id'].tolist() == sorted(nombres_documentos)[:10]
for consulta_vacia in ['', 'zzzzterminoinexistentexxxxx']:
    assert not set(tokenizar(consulta_vacia)) & df_terminos.keys()
    assert obtener_ranking(puntuar_tfidf(consulta_vacia)).empty
    assert obtener_ranking(puntuar_bm25(consulta_vacia)).empty
assert np.allclose(puntuar_bm25('sea sea ship'), puntuar_bm25('sea ship'))
for qid in consultas:
    for ranking in [rankings_tfidf[qid], rankings_bm25[qid]]:
        assert len(ranking) <= 10 and (ranking['Score'] > 0).all()
        assert ranking['doc_id'].is_unique
    assert np.isfinite(scores_tfidf[qid]).all() and np.isfinite(scores_bm25[qid]).all()
    assert (scores_tfidf[qid] >= -1e-12).all() and (scores_tfidf[qid] <= 1 + 1e-12).all()
for i in indices_elegidos:
    assert np.isclose(sum(aporte_bm25(t, i) for t in terminos_elegidos),
                      scores_bm25[consulta_elegida][i])
print('Comprobaciones correctas.')


Comprobaciones correctas.


**Uso de IA:** se utilizó IA como apoyo para implementar y comprobar los cálculos,
organizar las comparaciones y redactar explicaciones. Las salidas guardadas se obtienen
al ejecutar el código sobre el corpus local. Como estudiante debo revisar el trabajo,
reproducirlo en mi entorno y poder explicar las decisiones y los resultados.